# 10 — Monitoring pack (the Board-style MI dashboard)

**Plain English:** The final step assembles `outputs/reports/monitoring_pack.md`
from the real outputs of every prior notebook. It is built to read as a **Board
monitoring pack**: it **opens with a RAG dashboard** tied to the appetite limits
(notebook 07) and an **actions table** for anything amber/red, then lays out the
supporting evidence — each metric labelled **leading or lagging** — and closes
with the governance, stress and disclosure notes.

> Format only — illustrative, **not a regulatory submission**.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
import monitor as m
print("monitor library loaded — vintages:", m.VINTAGES)

monitor library loaded — vintages: ['2007', '2008', '2015']


In [2]:
# Assemble the Board-style monitoring pack from real outputs -------------
T = m.OUT_TABLES
rep = m.REPO_ROOT / "outputs" / "reports"
rep.mkdir(parents=True, exist_ok=True)

appetite = pd.read_csv(T / "07_appetite_status.csv")
trend    = pd.read_csv(T / "07_leading_trends.csv")
stress   = pd.read_csv(T / "07_stress_vs_limits.csv")
probs_b  = pd.read_csv(T / "02_bucket_transition_matrix.csv", index_col=0)
rr       = pd.read_csv(T / "02_roll_rates.csv")
moves    = pd.read_csv(T / "03_stage_movements.csv")
wsumm    = pd.read_csv(T / "04_watchlist_summary.csv")
vint     = pd.read_csv(T / "05_vintage_cumulative_default.csv")
state    = pd.read_csv(T / "06_concentration_state.csv", index_col=0)
hhi_tbl  = pd.read_csv(T / "06_concentration_hhi.csv")
lvr      = pd.read_csv(T / "06_concentration_lvr.csv", index_col=0)
occ      = pd.read_csv(T / "06_concentration_occupancy.csv", index_col=0)
purpose  = pd.read_csv(T / "06_concentration_purpose.csv", index_col=0)
chan     = pd.read_csv(T / "06_channel_performance.csv")
mod_view = pd.read_csv(T / "08_modified_exposure.csv")
scal     = pd.read_csv(T / "08_collections_scalability.csv")
psi_tbl  = pd.read_csv(T / "09_psi.csv")
grade    = pd.read_csv(T / "09_realised_default_by_grade.csv", index_col=0)
util     = pd.read_csv(T / "07_limit_utilisation.csv")
ecl      = pd.read_csv(T / "07_ecl_provisions.csv")
cur_lvr  = pd.read_csv(T / "06_concentration_current_lvr.csv", index_col=0)
topn     = pd.read_csv(T / "06_concentration_topn.csv")
wf       = pd.read_csv(T / "04_watchlist_workflow.csv")
tsumm    = pd.read_csv(T / "04_watchlist_triggers.csv")
hardship = pd.read_csv(T / "08_hardship_summary.csv")
new_conc = pd.read_csv(T / "08_new_concessions_by_year.csv")
utp_tbl  = pd.read_csv(T / "08_utp_overlay.csv")
pred     = pd.read_csv(T / "09_validation_predictiveness.csv")

worst = appetite.RAG.map({"GREEN": 0, "AMBER": 1, "RED": 2, "n/a": 0}).max()
overall = {0: "GREEN", 1: "AMBER", 2: "RED"}[int(worst)]

L = []
def add(*xs): L.extend(xs)

add("# Portfolio Monitoring Pack — loan-level (Freddie Mac SFLLD)\n")
add("_Real loan-level mortgage data. The monitoring mechanics apply equally to "
    "commercial loan portfolios with a monthly status feed._\n")
add("_Format only — illustrative, **not a regulatory submission**._\n")

# --- 1. RAG dashboard (MON-2) ------------------------------------------------
add("## 1. Risk appetite dashboard — RAG vs limits\n")
add(f"**Overall portfolio status: {overall}.** Each metric is scored against the "
    "amber (early-warning) and red (limit) thresholds in `config/risk_appetite.yaml` "
    "(APS 220 paras 20/35; APG 220 para 65). `type` flags leading vs lagging.\n")
dash = appetite[["metric", "type", "last_period", "this_period", "amber", "red (limit)", "RAG"]]
add(dash.to_markdown(index=False) + "\n")

# Actions table for anything amber/red
flagged = appetite[appetite.RAG.isin(["AMBER", "RED"])]
add("### Actions (amber/red)\n")
if len(flagged):
    act = flagged[["metric", "RAG", "breach_action", "owner", "review_cycle"]].rename(
        columns={"breach_action": "action", "review_cycle": "due"})
    add(act.to_markdown(index=False) + "\n")
else:
    add("_All metrics within appetite this period — no escalations required._\n")
add("**Forward-looking view:** leading indicators (Stage 2 share, roll rates, SICR) "
    "are read first because they move before losses; the vintage curves (section 7) show "
    "how fast a downturn cohort can deteriorate, and the stress test (section 11) shows "
    "the same metrics against their limits under graded downturn scenarios.\n")
add("**Appetite vs tolerance (APS 220 para 20):** the *amber* level is the risk appetite "
    "(an early warning); the *red* level is the risk tolerance (a hard limit). Amber -> "
    "investigate; red -> escalate to the Board Risk Committee. The appetite statement is "
    "reviewed at least annually (see `config/risk_appetite.yaml`).\n")

# --- 1b. IFRS 9 ECL / provision coverage (APG 220 para 67(b)) ---------------
add("### Provisions & expected loss (APG 220 para 67(b))\n")
add("Illustrative IFRS 9 ECL from stage exposures x config coverage rates; the EL rate "
    "and NPL coverage ratio also drive the `loss_rate` / `provision_coverage` RAG metrics.\n")
add(ecl.to_markdown(index=False) + "\n")

# --- 1c. Limit utilisation / headroom (daily-layer analog) ------------------
add("### Limit utilisation & headroom\n")
add("How much of each limit is used at the reporting date, and the headroom remaining. "
    "This is the MONTHLY analog of the DAILY facility/limit-excess monitoring APS 113 "
    "Att. D (EAD para 6) / Basel CRE36.92 require — the daily layer is described in "
    "`docs/governance.md`.\n")
add(util.to_markdown(index=False) + "\n")

# --- 2. Leading-indicator trends (MON-3) ------------------------------------
add("## 2. Leading-indicator trends (forward-looking)\n")
add("APG 220 para 66 — do not rely solely on lagging arrears. Trailing-12m roll rates "
    "and the SICR (Current->Stage 2) migration rate, tracked over time:\n")
add(trend.to_markdown(index=False) + "\n")

# --- 3. Transition matrix + roll rates (lagging/leading labelled) -----------
add("## 3. Monthly delinquency-bucket transition matrix  _(lagging)_\n")
add(probs_b.round(4).to_markdown() + "\n")
add("![heatmap](../charts/02_bucket_transition_heatmap.png)\n")
add("## 4. Headline roll rates  _(leading — deterioration moves before default)_\n")
add(rr.to_markdown(index=False) + "\n")

# --- 5. Stage movements / 6. watchlist / 7. vintage -------------------------
add("## 5. IFRS 9 stage movements (loan-months)  _(mixed)_\n")
add(moves.to_markdown(index=False) + "\n")
add("## 6. Early-warning watchlist (by vintage / stage)  _(leading)_\n")
add(wsumm.to_markdown(index=False) + "\n")
add("**Trigger mix** — the early-warning criterion that raised each watchlist item "
    "(APS 220 para 33; APG 220 para 63):\n")
add(tsumm.to_markdown(index=False) + "\n")
add("**Problem-exposure workflow** — each trigger maps to an action, an accountable "
    "owner and an escalation timeframe (config: `watchlist_workflow`):\n")
add(wf.to_markdown(index=False) + "\n")
add("## 7. Vintage tracking — cumulative default by months on book  _(lagging)_\n")
add(vint.to_markdown(index=False) + "\n")
add("![vintage curves](../charts/05_vintage_default_curves.png)\n")

# --- 8. Concentration: state + HHI + LVR + product + channel (MON-4) ---------
add("## 8. Concentration — geography, HHI, high-LVR, product mix & channel (APS 220 paras 35/39)\n")
add("_Format only — illustrative, not a regulatory submission._\n")
add("**By state (top 10):**\n")
add(state.head(10).round(2).to_markdown() + "\n")
add("**Geographic HHI:**\n")
add(hhi_tbl.to_markdown(index=False) + "\n")
add("**High-LVR concentration (by original LVR band):**\n")
add(lvr.round(2).to_markdown() + "\n")
add("**Product mix — occupancy (higher-risk product; APS 220 para 35(b)):** investor "
    "lending is watched separately from owner-occupier.\n")
add(occ.round(2).to_markdown() + "\n")
add("**Product mix — loan purpose:** cash-out refinances carry higher risk than purchase "
    "or no-cash-out refinances.\n")
add(purpose.round(2).to_markdown() + "\n")
add("**Acquisition channel — third-party-originator monitoring (APS 220 para 39; "
    "APG 220 paras 307-308):** broker / correspondent / TPO loans are originated away from "
    "the lender's own desk, so their lifetime default experience is tracked separately from "
    "retail.\n")
add(chan.to_markdown(index=False) + "\n")
add("**Current / indexed-LVR concentration (continuous collateral monitoring; Basel "
    "CRE36.140):** origination LVR is static, so the live collateral view marks each "
    "property to market via an illustrative HPI path — the marked-to-market counterpart of "
    "the original-LVR table above.\n")
add(cur_lvr.round(2).to_markdown() + "\n")
add("**Single-name / large-exposure concentration (APG 220 paras 77-80):** immaterial for "
    "a granular retail mortgage pool — reported so the dimension is not silently omitted.\n")
add(topn.to_markdown(index=False) + "\n")

# --- 9. Problem exposures (MON-5) -------------------------------------------
add("## 9. Problem exposures — modifications & collections scalability (APS 220 para 79)\n")
add("Modified / restructured loans and whether they cured or re-defaulted:\n")
add(mod_view.to_markdown(index=False) + "\n")
add("Collections scalability — trough vs crisis-peak monthly arrears (30+DPD) rate; "
    "the surge multiple is the load the workout function must be able to absorb:\n")
add(scal.to_markdown(index=False) + "\n")
add("**Hardship / concession outcomes incl. ultimate loss (APG 220 para 68):** cure, "
    "re-default and the share that reached an actual Default credit event (ultimate-loss "
    "proxy):\n")
add(hardship.to_markdown(index=False) + "\n")
add("**Trend in new concession requests by year & product:** (the SFLLD records granted "
    "concessions only, so an approval rate is not observable):\n")
add(new_conc.to_markdown(index=False) + "\n")
add("**Unlikely-to-pay (UTP) overlay (APS 220 default definition):** Stage 3 uses the "
    "90-DPD/credit-event backstop only; this flags ever-modified loans still performing as "
    "an illustrative UTP early-warning (production would fold UTP into the default "
    "definition):\n")
add(utp_tbl.to_markdown(index=False) + "\n")

# --- 10. Model performance / PSI (MON-6) ------------------------------------
add("## 10. Model performance — population stability (PSI) & backtest feed\n")
add("Layer 4 (rating-system performance). PSI of origination features vs the calm-2015 "
    "reference, and realised default by grade — the backtest feed for the sister "
    "[mortgage-credit-risk-pd-lgd-ead](https://github.com/Jane511/mortgage-credit-risk-pd-lgd-ead) model:\n")
add(psi_tbl.to_markdown(index=False) + "\n")
add("Realised cumulative default (%) by credit-score grade x vintage:\n")
add(grade.to_markdown() + "\n")
add("**Validation — is the leading indicator predictive? (APG 113 para 140, element 3 "
    "'Performance'):** loans that entered Stage 2 (SICR) within their first 12 months "
    "default at a far higher rate than those that did not — evidence the watch trigger "
    "leads losses rather than lagging them. Full 8-element validation in `docs/validation.md`.\n")
add(pred.to_markdown(index=False) + "\n")

# --- 11. Governance / stress / disclosure notes (MON-7/8/9) -----------------
add("## 11. Governance, stress & disclosure notes\n")
add("**Stress -> limits (MON-7; APS 220 paras 73/76).** Per-metric downturn multipliers "
    "(config) re-test the flow/quality metrics against their limits under two graded "
    "scenarios — watch/roll FLOW metrics move more than the NPL STOCK, grounded in this "
    "repo's 2007-vs-2015 vintage ratios. Multipliers are illustrative and would be "
    "independently validated before use (APS 220 para 76):\n")
add(stress.to_markdown(index=False) + "\n")
add("**Independence (Basel CRE36.57 / APS 220 para 28).** Monitoring and the appetite "
    "limits are owned by an independent 2nd-line Credit Risk function, functionally separate "
    "from mortgage origination; remediation actions are executed by the business but "
    "overseen by Credit Risk. See `docs/governance.md`.\n")
add("**Reporting cadence (Step 12).** Front-line: daily/weekly limit-excess & arrears flow; "
    "Credit Risk Committee: monthly watchlist, roll rates, trigger mix; Board Risk Committee: "
    "monthly appetite RAG, concentration, provisions; model governance: at least annually "
    "(PSI / validation); Audit Committee: quarterly independent assurance. Full forum table "
    "in `docs/governance.md`.\n")
add("**Independent validation (MON-8; APG 113 para 140).** The monitoring framework is "
    "subject to the 8-element validation framework, evidenced by the predictiveness test in "
    "section 10 (element 3) and the data-quality reconciliation (nb 00). Full 8-element "
    "assessment in `docs/validation.md`; the framework would be independently validated "
    "annually. _Demo, not a production system._\n")
add("**Scope (Step 5 / sections 4.3-4.4).** Rating-refresh cadence (annual; Basel CRE36.41) "
    "lives in the sister PD/LGD/EAD model — a PSI breach here (section 10) triggers a refresh. "
    "Country/transfer risk (single-country US book) and purchased-receivables seller/servicer "
    "monitoring are out of scope for this own-book demo (see `docs/governance.md`).\n")
add("**APS 330 / Pillar 3 framing (MON-9).** The concentration and credit-quality outputs "
    "(sections 7-8) are the inputs that feed **Pillar 3 (APS 330)** credit-risk disclosure. "
    "Any APS 330-style table here is **format only — illustrative, not a regulatory submission**.\n")

(rep / "monitoring_pack.md").write_text("\n".join(L), encoding="utf-8")
print("wrote ->", rep / "monitoring_pack.md", f"| overall RAG: {overall}")

wrote -> D:\Jane\Job Search\Github\bank\github project\mortgage-portfolio-monitoring\outputs\reports\monitoring_pack.md | overall RAG: GREEN
